# Function Testing Notebook

Author: Pete King

This notebook tests custom functions developed in the various helper modules to verify proper operation.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import yfinance as yf

import data_prep as dp

DATA_FILENAME='etf_raw_data.csv'
ETF='SPY'

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Import and inspect ETF price data

In [2]:
df = pd.read_csv(
    DATA_FILENAME,
    index_col='date',
    parse_dates=True
)
df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.175379,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.347328,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.398905,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.656818,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.759996,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-27,91.366028,72.861801,414.700012,78.340057,94.285789,243.100006,107.179611,562.580017,634.090027,109.669998,...,48.910000,62.560001,47.810001,159.199997,129.919998,81.779999,40.009998,45.590000,143.259995,105.680000
2026-03-30,91.375999,73.230545,414.579987,78.429619,94.953560,239.610001,107.866783,558.280029,631.969971,110.349998,...,49.090000,61.959999,48.360001,156.610001,127.500000,81.879997,40.209999,45.919998,143.820007,105.660004
2026-03-31,91.375999,73.389999,430.290009,79.175995,95.123001,248.000000,108.543999,577.179993,650.340027,110.360001,...,49.970001,61.259998,49.369999,161.729996,132.899994,81.980003,40.830002,45.889999,146.610001,108.980003


## Compute and display daily return

Here we test the ability of the log_return function to compute daily returns and inspect the results.

In [3]:
test_df = df[[ETF]]
etf_return = test_df['SPY'].rolling(2).apply(dp.log_return, raw=True)
test_df[ETF + '_return'] = etf_return.values
test_df

,SPY,SPY_return
date,,
1993-01-29,24.175379,NaN
1993-02-01,24.347328,0.007087
1993-02-02,24.398905,0.002116
1993-02-03,24.656818,0.010515
1993-02-04,24.759996,0.004176
...,...,...
2026-03-27,634.090027,-0.017199
2026-03-30,631.969971,-0.003349
2026-03-31,650.340027,0.028653


In [4]:
chart = alt.Chart(test_df.dropna().reset_index()).mark_circle(size=10).encode(
    x='date:T',
    y=ETF + '_return:Q'
)
chart.properties(height=200, width=800)

alt.Chart(...)

## Discussion

From the chart we can see that the mean of daily returns appears to be nearly zero -- an empirical justification of the zero mean assumption for expected return (E\[R\]).

Since volatility for an asset is a measure of the deviation of returns from expected return, we can get a feel for an asset's volatility just by inspecting the plot.  We see a general trend of baseline low volatility (for example, from Jan 2004 to Jan 2007), with periods of high volatility that tend to gradually revert to baseline (for example during the 'Great Recession', from late 2007, spiking in late 2008 / early 2009, and gradually reverting to a lower baseline by roughly 2012).

In [5]:
etf_return.describe()

count    8350.000000
mean        0.000395
std         0.011725
min        -0.115887
25%        -0.004344
50%         0.000678
75%         0.005918
max         0.135578
Name: SPY, dtype: float64

The main idea with this project is to think of the daily return for each asset as a random variable (R), and then investigate its statistical properties.  

***Right away, from this simple statistical description (above), we can get an idea of what to expect for the properties of an asset's returns (R):***
 - Estimated **expected return** (E\[R\]): 0.04 percent (very close to zero)
 - Estimated long-term (baseline) **volatility**: 1.17 percent

*Note that the financial term "volatility" can have mean interpretations, but here we mean the long-term standard deviation of return (R), assuming zero mean.*

In [6]:
# Compute using the zero-mean assumption for expected return
vol = np.sqrt(
    np.sum(etf_return.dropna().values**2) / len(etf_return.dropna())
)
print(f'Estimated long-term volatility with zero-mean assumption: \
        {vol * 100:2.2f} percent'
     )

Estimated long-term volatility with zero-mean assumption:         1.17 percent
